In [51]:
import pandas as pd
from pathlib import Path

In [52]:
input_dir = Path('./data')
output_dir = Path('./data/clean')

In [53]:
countries = pd.read_csv(input_dir / 'WB_WDI_ST_INT_ARVL_WIDEF.csv')
numeric_cols = [col for col in countries.columns if col.isdigit()]
countries = countries[["REF_AREA_LABEL"] + numeric_cols]
countries[numeric_cols] = countries[numeric_cols].apply(lambda x: x.lower() if isinstance(x, str) else x).apply(pd.to_numeric, errors='coerce').fillna(-1).astype(int).replace(-1, pd.NA)
countries.rename(columns={"REF_AREA_LABEL": "name"}, inplace=True)
countries.head()

,name,1995,1996,1997,1998,1999,2000,2001,2002,2003,...,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
0,Guyana,106000,92000,76000,66000,75000,105000,99000,104000,101000,...,157000,177000,200000,206000,207000,235000,247000,287000,315000,86400
1,Other small states,16652119,16883667,18570111,19110481,20547389,22028654,22699583,23251643,23525106,...,33747412,36199883,39516963,42941757,43653648,46851127,49979552,51524462,51930012,<NA>
2,South Africa,4684000,5186000,5170000,5898000,6026000,6001000,5908000,6550000,6640000,...,12097000,13069000,14318000,14530000,13952000,15121000,14975000,15004000,14797000,3886600
3,Central African Republic,26000,21000,17000,7500,10000,11199,9899,2900,5699,...,65400,70800,83500,95699,120500,82000,107000,109000,87000,<NA>
4,Switzerland,6946000,6730000,7039000,7185000,7154000,7821000,7455000,6868000,6530000,...,8534000,8566000,8967000,9158000,9305000,10402000,11133000,11715000,11818000,<NA>


In [54]:
heritage = pd.read_csv(input_dir / 'unesco_world_heritage' / 'archive' / 'whc-sites-2023.csv')
heritage = heritage[["name_en", "short_description_en", "danger", "latitude", "longitude", "category", "states_name_en", "region_en"]]
heritage.rename(columns={"name_en": "name", "short_description_en": "short_description", "states_name_en": "states_name", "region_en": "region"}, inplace=True)
heritage.head()

,name,short_description,danger,latitude,longitude,category,states_name,region
0,Cultural Landscape and Archaeological Remains ...,<p>The cultural landscape and archaeological r...,1,34.846940,67.825250,Cultural,Afghanistan,Asia and the Pacific
1,Minaret and Archaeological Remains of Jam,"<p>The 65m-tall Minaret of Jam is a graceful, ...",1,34.396417,64.515889,Cultural,Afghanistan,Asia and the Pacific
2,Historic Centres of Berat and Gjirokastra,<p>Berat and Gjirokastra are inscribed as rare...,0,40.074167,20.140833,Cultural,Albania,Europe and North America
3,Butrint,"<p>Inhabited since prehistoric times, Butrint ...",0,39.745732,20.020950,Cultural,Albania,Europe and North America
4,Al Qal'a of Beni Hammad,<p>In a mountainous site of extraordinary beau...,0,35.818440,4.786840,Cultural,Algeria,Arab States


In [55]:
cities = pd.read_csv(input_dir / 'worldwide_travel_dataset' / 'Worldwide Travel Cities Dataset (Ratings and Climate).csv')
cities = cities[["city", "country", "avg_temp_monthly", "ideal_durations", "budget_level", "culture", "adventure", "nature", "beaches", "nightlife", "cuisine", "wellness", "urban", "seclusion"]]
cities.head()

,city,country,avg_temp_monthly,ideal_durations,budget_level,culture,adventure,nature,beaches,nightlife,cuisine,wellness,urban,seclusion
0,Milan,Italy,"{""1"":{""avg"":3.7,""max"":7.8,""min"":0.4},""2"":{""avg...","[""Short trip"",""One week""]",Luxury,5,2,2,1,4,5,3,5,2
1,Yasawa Islands,Fiji,"{""1"":{""avg"":28,""max"":30.8,""min"":25.8},""2"":{""av...","[""Long trip"",""One week""]",Luxury,2,4,5,5,2,3,4,1,5
2,Whistler,Canada,"{""1"":{""avg"":-2.5,""max"":0.4,""min"":-5.5},""2"":{""a...","[""Short trip"",""Weekend"",""One week""]",Luxury,3,5,5,2,3,3,4,2,4
3,Guanajuato,Mexico,"{""1"":{""avg"":15.5,""max"":22.8,""min"":8.7},""2"":{""a...","[""Weekend"",""One week"",""Short trip""]",Mid-range,5,3,3,1,3,4,3,4,2
4,Surabaya,Indonesia,"{""1"":{""avg"":28.1,""max"":32.5,""min"":25.5},""2"":{""...","[""Short trip"",""Weekend""]",Budget,4,3,3,2,3,4,3,4,2


Standardize country names

In [56]:
inv_mapping = {'Bahamas': 'Bahamas, The','Cape Verde': 'Cabo Verde',
 'Cook Islands': None,
 'Czech Republic': 'Czechia',
 'Egypt': 'Egypt, Arab Rep.',
 'Equatorial Guinea': None,
 'French Guiana': None,
 'Gambia': 'Gambia, The',
 'Greenland': None,
 'Hong Kong': 'Hong Kong SAR, China',
 'Iran': 'Iran, Islamic Rep.',
 'Ivory Coast': 'Côte d’Ivoire',
 'Kosovo': None,
 'Kyrgyzstan': 'Kyrgyz Republic',
 'Laos': 'Lao PDR',
 'Republic of the Congo': 'Congo, Rep.',
 'Russia': 'Russian Federation',
 'Réunion': None,
 'Saint Kitts and Nevis': 'St. Kitts and Nevis',
 'Saint Lucia': 'St. Lucia',
 'Slovakia':'Slovak Republic',
 'South Korea': 'Korea, Rep.',
 'Taiwan': None,
 'Turkey': 'Türkiye',
 'Turks and Caicos': 'Turks and Caicos Islands',
 'U.S. Virgin Islands': 'Virgin Islands (U.S.)',
 'Venezuela': 'Venezuela, RB'}
country_map = {v: k for k, v in inv_mapping.items() if v is not None}

city_map = {'Quebec': 'Canada'}

countries['name'] = countries['name'].replace(country_map)
cities['country'] = cities['country'].replace(city_map)
heritage['states_name'] = heritage['states_name'].replace(country_map)

Add iso country codes to match with geoJSON

source: https://gist.github.com/themeteorchef/dcffd74ca3ab45277c81

In [57]:
iso_countries = {
    'AF' : 'Afghanistan',
    'AX' : 'Aland Islands',
    'AL' : 'Albania',
    'DZ' : 'Algeria',
    'AS' : 'American Samoa',
    'AD' : 'Andorra',
    'AO' : 'Angola',
    'AI' : 'Anguilla',
    'AQ' : 'Antarctica',
    'AG' : 'Antigua and Barbuda',
    'AR' : 'Argentina',
    'AM' : 'Armenia',
    'AW' : 'Aruba',
    'AU' : 'Australia',
    'AT' : 'Austria',
    'AZ' : 'Azerbaijan',
    'BS' : 'Bahamas',
    'BH' : 'Bahrain',
    'BD' : 'Bangladesh',
    'BB' : 'Barbados',
    'BY' : 'Belarus',
    'BE' : 'Belgium',
    'BZ' : 'Belize',
    'BJ' : 'Benin',
    'BM' : 'Bermuda',
    'BT' : 'Bhutan',
    'BO' : 'Bolivia',
    'BA' : 'Bosnia and Herzegovina',
    'BW' : 'Botswana',
    'BV' : 'Bouvet Island',
    'BR' : 'Brazil',
    'IO' : 'British Indian Ocean Territory',
    'BN' : 'Brunei Darussalam',
    'BG' : 'Bulgaria',
    'BF' : 'Burkina Faso',
    'BI' : 'Burundi',
    'KH' : 'Cambodia',
    'CM' : 'Cameroon',
    'CA' : 'Canada',
    'CV' : 'Cape Verde',
    'KY' : 'Cayman Islands',
    'CF' : 'Central African Republic',
    'TD' : 'Chad',
    'CL' : 'Chile',
    'CN' : 'China',
    'CX' : 'Christmas Island',
    'CC' : 'Cocos (Keeling) Islands',
    'CO' : 'Colombia',
    'KM' : 'Comoros',
    'CG' : 'Republic of the Congo',
    'CD' : 'Congo, Dem. Rep.',
    'CK' : 'Cook Islands',
    'CR' : 'Costa Rica',
    'CI' : 'Ivory Coast',
    'HR' : 'Croatia',
    'CU' : 'Cuba',
    'CY' : 'Cyprus',
    'CZ' : 'Czech Republic',
    'DK' : 'Denmark',
    'DJ' : 'Djibouti',
    'DM' : 'Dominica',
    'DO' : 'Dominican Republic',
    'EC' : 'Ecuador',
    'EG' : 'Egypt',
    'SV' : 'El Salvador',
    'GQ' : 'Equatorial Guinea',
    'ER' : 'Eritrea',
    'EE' : 'Estonia',
    'ET' : 'Ethiopia',
    'FK' : 'Falkland Islands (Malvinas)',
    'FO' : 'Faroe Islands',
    'FJ' : 'Fiji',
    'FI' : 'Finland',
    'FR' : 'France',
    'GF' : 'French Guiana',
    'PF' : 'French Polynesia',
    'TF' : 'French Southern Territories',
    'GA' : 'Gabon',
    'GM' : 'Gambia',
    'GE' : 'Georgia',
    'DE' : 'Germany',
    'GH' : 'Ghana',
    'GI' : 'Gibraltar',
    'GR' : 'Greece',
    'GL' : 'Greenland',
    'GD' : 'Grenada',
    'GP' : 'Guadeloupe',
    'GU' : 'Guam',
    'GT' : 'Guatemala',
    'GG' : 'Guernsey',
    'GN' : 'Guinea',
    'GW' : 'Guinea-Bissau',
    'GY' : 'Guyana',
    'HT' : 'Haiti',
    'HM' : 'Heard Island & Mcdonald Islands',
    'VA' : 'Holy See (Vatican City State)',
    'HN' : 'Honduras',
    'HK' : 'Hong Kong',
    'HU' : 'Hungary',
    'IS' : 'Iceland',
    'IN' : 'India',
    'ID' : 'Indonesia',
    'IR' : 'Iran',
    'IQ' : 'Iraq',
    'IE' : 'Ireland',
    'IM' : 'Isle Of Man',
    'IL' : 'Israel',
    'IT' : 'Italy',
    'JM' : 'Jamaica',
    'JP' : 'Japan',
    'JE' : 'Jersey',
    'JO' : 'Jordan',
    'KZ' : 'Kazakhstan',
    'KE' : 'Kenya',
    'KI' : 'Kiribati',
    'KR' : 'South Korea',
    'KW' : 'Kuwait',
    'KG' : 'Kyrgyzstan',
    'LA' : 'Laos',
    'LV' : 'Latvia',
    'LB' : 'Lebanon',
    'LS' : 'Lesotho',
    'LR' : 'Liberia',
    'LY' : 'Libya',
    'LI' : 'Liechtenstein',
    'LT' : 'Lithuania',
    'LU' : 'Luxembourg',
    'MO' : 'Macao SAR, China',
    'MK' : 'North Macedonia',
    'MG' : 'Madagascar',
    'MW' : 'Malawi',
    'MY' : 'Malaysia',
    'MV' : 'Maldives',
    'ML' : 'Mali',
    'MT' : 'Malta',
    'MH' : 'Marshall Islands',
    'MQ' : 'Martinique',
    'MR' : 'Mauritania',
    'MU' : 'Mauritius',
    'YT' : 'Mayotte',
    'MX' : 'Mexico',
    'FM' : 'Micronesia, Fed. Sts.',
    'MD' : 'Moldova',
    'MC' : 'Monaco',
    'MN' : 'Mongolia',
    'ME' : 'Montenegro',
    'MS' : 'Montserrat',
    'MA' : 'Morocco',
    'MZ' : 'Mozambique',
    'MM' : 'Myanmar',
    'NA' : 'Namibia',
    'NR' : 'Nauru',
    'NP' : 'Nepal',
    'NL' : 'Netherlands',
    'AN' : 'Netherlands Antilles',
    'NC' : 'New Caledonia',
    'NZ' : 'New Zealand',
    'NI' : 'Nicaragua',
    'NE' : 'Niger',
    'NG' : 'Nigeria',
    'NU' : 'Niue',
    'NF' : 'Norfolk Island',
    'MP' : 'Northern Mariana Islands',
    'NO' : 'Norway',
    'OM' : 'Oman',
    'PK' : 'Pakistan',
    'PW' : 'Palau',
    'PS' : 'Palestinian Territory, Occupied',
    'PA' : 'Panama',
    'PG' : 'Papua New Guinea',
    'PY' : 'Paraguay',
    'PE' : 'Peru',
    'PH' : 'Philippines',
    'PN' : 'Pitcairn',
    'PL' : 'Poland',
    'PT' : 'Portugal',
    'PR' : 'Puerto Rico',
    'QA' : 'Qatar',
    'RE' : 'Reunion',
    'RO' : 'Romania',
    'RU' : 'Russia',
    'RW' : 'Rwanda',
    'BL' : 'Saint Barthelemy',
    'SH' : 'Saint Helena',
    'KN' : 'Saint Kitts and Nevis',
    'LC' : 'Saint Lucia',
    'MF' : 'Saint Martin',
    'PM' : 'Saint Pierre and Miquelon',
    'VC' : 'St. Vincent and Grenadines',
    'WS' : 'Samoa',
    'SM' : 'San Marino',
    'ST' : 'São Tomé and Príncipe',
    'SA' : 'Saudi Arabia',
    'SN' : 'Senegal',
    'RS' : 'Serbia',
    'SC' : 'Seychelles',
    'SL' : 'Sierra Leone',
    'SG' : 'Singapore',
    'SK' : 'Slovakia',
    'SI' : 'Slovenia',
    'SB' : 'Solomon Islands',
    'SO' : 'Somalia',
    'ZA' : 'South Africa',
    'GS' : 'South Georgia and Sandwich Isl.',
    'ES' : 'Spain',
    'LK' : 'Sri Lanka',
    'SD' : 'Sudan',
    'SR' : 'Suriname',
    'SJ' : 'Svalbard and Jan Mayen',
    'SZ' : 'Swaziland',
    'SE' : 'Sweden',
    'CH' : 'Switzerland',
    'SY' : 'Syrian Arab Republic',
    'TW' : 'Taiwan',
    'TJ' : 'Tajikistan',
    'TZ' : 'Tanzania',
    'TH' : 'Thailand',
    'TL' : 'Timor-Leste',
    'TG' : 'Togo',
    'TK' : 'Tokelau',
    'TO' : 'Tonga',
    'TT' : 'Trinidad and Tobago',
    'TN' : 'Tunisia',
    'TR' : 'Turkey',
    'TM' : 'Turkmenistan',
    'TC' : 'Turks and Caicos',
    'TV' : 'Tuvalu',
    'UG' : 'Uganda',
    'UA' : 'Ukraine',
    'AE' : 'United Arab Emirates',
    'GB' : 'United Kingdom',
    'US' : 'United States',
    'UM' : 'United States Outlying Islands',
    'UY' : 'Uruguay',
    'UZ' : 'Uzbekistan',
    'VU' : 'Vanuatu',
    'VE' : 'Venezuela',
    'VN' : 'Vietnam',
    'VG' : 'British Virgin Islands',
    'VI' : 'U.S. Virgin Islands',
    'WF' : 'Wallis and Futuna',
    'EH' : 'Western Sahara',
    'YE' : 'Yemen, Rep.',
    'ZM' : 'Zambia',
    'ZW' : 'Zimbabwe'
}
iso_countries = {v: k for k, v in iso_countries.items()}

countries['iso_code'] = (countries['name']
                         .map(iso_countries)
                         .apply(lambda x: x.lower() if isinstance(x, str) else x))

In [58]:

output_dir.mkdir(parents=True, exist_ok=True)

cities.to_csv(output_dir / 'cities.csv', index=False)
countries.to_csv(output_dir / 'countries.csv', index=False)
heritage.to_csv(output_dir / 'heritage.csv', index=False)